# Step 4 — Map & traffic validation (safe first run)


In [8]:
import sys
from IPython.display import display
from routing.config import ORIGIN, DESTINATION
from routing.map_loader import load_road_graph
from routing.map_view import interactive_map
from routing.traffic_api import TomTomClient
from routing.traffic_pipeline import collect_traffic_snapshot, make_routing_graph, preview_probes, save_summary
print('Python:', sys.executable)

Python: /Users/mojtabashabani/wiser/mary/.venv/bin/python


In [9]:

RADIUS_M = 1200
G = load_road_graph(radius_m=RADIUS_M)
print('Graph:', type(G).__name__, G.number_of_nodes(), 'nodes,', G.number_of_edges(), 'edges')
assert G.is_directed()
assert all(float(d['length']) >= 0 for *_, d in G.edges(keys=True, data=True))
display(preview_probes(G, limit=5))

TypeError: load_road_graph() got an unexpected keyword argument 'radius_m'

In [ ]:
# No HTTP request here. Use this map for testing Origin and Destination.
basic_widget, basic_state = interactive_map(G, origin=ORIGIN, destination=DESTINATION)
display(basic_widget)
print('Double-click on the map, then run the NEXT cell.')

NameError: name 'G' is not defined

In [ ]:
print('Origin node:', basic_state['origin_node'])
print('Destination node:', basic_state['destination_node'])
print('Last click:', basic_state['last_click_latlon'])
if basic_state['last_click_latlon'] is None:
    print('No double-click yet; map must render and receive an interaction.')
else:
    assert basic_state['origin_node'] in G
    assert basic_state['destination_node'] in G
    print('PASS — interactive origin')

In [ ]:
# SAFE DEFAULT: 0 live API calls. Flip to True consciously.
RUN_LIVE_TRAFFIC = False
REQUEST_LIMIT = 12

if RUN_LIVE_TRAFFIC:
    with TomTomClient() as client:
        snapshot, summary = collect_traffic_snapshot(
            G, client, max_requests=REQUEST_LIMIT,
        )
    save_summary(summary)   # summary only, no raw TomTom responses
    print('LIVE TomTom snapshot collected')
else:
    # max_requests=0: does not call the client (no key needed here).
    snapshot, summary = collect_traffic_snapshot(G, client=None, max_requests=0)
    print('OFFLINE fallback-only snapshot: ZERO TomTom requests')

print('Requests:', summary['requests_made'])
print('Usable coverage (%):', summary['usable_coverage_percent'])
print('Usable among sampled (%):', summary['usable_among_sampled_percent'])
print('Counts:', summary['counts'])
print('Mapping diagnostics:', summary['map_matching_diagnostics'])
print('API error groups:', summary['api_error_groups'])

In [ ]:
# This map shows congestion/closures by default. "Unknown traffic" is a checkbox, OFF initially.
traffic_widget, traffic_state = interactive_map(
    G, origin=ORIGIN, destination=DESTINATION, traffic_graph=snapshot
)
display(traffic_widget)

In [ ]:
free_time_graph = make_routing_graph(snapshot, objective='time', scenario='free_flow')
measured_time_graph = make_routing_graph(snapshot, objective='time', scenario='measured')
measured_distance_graph = make_routing_graph(snapshot, objective='distance', scenario='measured')
print('Free-flow time edges:', free_time_graph.number_of_edges())
print('Measured time edges:', measured_time_graph.number_of_edges())
print('Measured distance edges:', measured_distance_graph.number_of_edges())
assert all(float(d['length']) >= 0 for *_, d in measured_time_graph.edges(keys=True, data=True))
print('PASS — routing graphs prepared; no search algorithm executed')